## 🎯 Learning Objectives
* Assess understanding of core ReAct loop components (Thought, Action, Observation).
* Evaluate ability to design and implement basic agentic planning mechanisms.
* Test proficiency in integrating and calling external tools within an agent.
* Reinforce knowledge of agent evaluation strategies through practical application.


# AG03-L13: Quiz - Building Agents from Scratch

**Course:** AG-03 — Building AI Agents from Scratch
**Section:** Testing and Hardening

## Exercise Task: Implement a ReAct Agent with a Tool

This quiz is designed to consolidate your understanding of the fundamental building blocks of an AI agent as covered in AG-03. Your task is to implement a simple ReAct agent that can answer questions about country capitals using a provided `CapitalFinderTool`.

### Requirements:
1.  **ReAct Loop Implementation:** Your agent must clearly demonstrate the `Thought`, `Action`, `Observation` loop. The agent should generate a thought, decide on an action (using the `CapitalFinderTool`), execute the action, and process the observation.
2.  **Agent Class:** Implement an `Agent` class that orchestrates this ReAct loop.
3.  **Tool Integration:** The agent must effectively use the pre-defined `CapitalFinderTool` to retrieve information.
4.  **Mock LLM Interaction:** Your agent will interact with a simplified `mock_llm_predict` function that simulates an LLM's response. You'll need to parse its output to extract thoughts and actions.
5.  **Answer a Specific Question:** The agent should be able to answer the question: "What is the capital of France?"

### Evaluation Criteria:
*   **Correctness:** Does the agent correctly identify the capital of France?
*   **ReAct Adherence:** Is the ReAct loop clearly implemented and followed?
*   **Code Clarity:** Is the code well-structured, readable, and commented?
*   **Robustness (Basic):** Does the agent handle the expected LLM output format correctly?

Good luck!


In [ ]:
import json
import re

# --- Setup Code: Mock LLM and Tool Definitions ---

# Mock knowledge base for the CapitalFinderTool
CAPITALS_DB = {
    "France": "Paris",
    "Germany": "Berlin",
    "Japan": "Tokyo",
    "Canada": "Ottawa",
    "Australia": "Canberra",
    "United States": "Washington D.C."
}

class CapitalFinderTool:
    """A tool to find the capital of a given country."""
    def __init__(self):
        self.name = "CapitalFinderTool"
        self.description = "Useful for finding the capital city of a country. Input should be a country name (string)."

    def run(self, country: str) -> str:
        """Executes the tool to find the capital."""
        print(f"\n[Tool Execution] Calling CapitalFinderTool with input: '{country}'")
        capital = CAPITALS_DB.get(country.strip(), "Unknown")
        if capital == "Unknown":
            return f"Could not find the capital for {country}."
        return f"The capital of {country} is {capital}."


def mock_llm_predict(prompt: str) -> str:
    """Simulates an LLM's response based on the prompt content.
    In a real scenario, this would be an API call to an actual LLM.
    """
    print(f"\n[LLM Call] Processing prompt:\n---\n{prompt}\n---")

    if "What is the capital of France?" in prompt and "CapitalFinderTool" in prompt:
        # Simulate LLM deciding to use the tool
        if "Observation" not in prompt:
            return (
                "Thought: The user is asking for the capital of France. I need to use the CapitalFinderTool to find this information.\n" +
                "Action: CapitalFinderTool\n" +
                "Action Input: France"
            )
        elif "The capital of France is Paris." in prompt:
            # Simulate LLM having the information and providing a final answer
            return (
                "Thought: I have successfully retrieved the capital of France, which is Paris. I can now provide the final answer.\n" +
                "Final Answer: The capital of France is Paris."
            )
    
    # Default or fallback response
    return "Thought: I am unable to answer this question with the available tools or information.\nFinal Answer: I don't know."


# Initialize the tool
capital_finder_tool = CapitalFinderTool()

# Available tools for the agent
TOOLS = {
    capital_finder_tool.name: capital_finder_tool
}

print("Setup complete: Mock LLM and CapitalFinderTool are ready.")


## Your Turn: Implement the ReAct Agent

Now, it's your turn to implement the `Agent` class. Your agent should:

1.  Take an initial `query`.
2.  Maintain a `scratchpad` to store the history of `Thought`, `Action`, `Observation` steps.
3.  Call the `mock_llm_predict` function with the current `scratchpad` to get the next step.
4.  Parse the LLM's response to identify `Thought`, `Action`, `Action Input`, or `Final Answer`.
5.  If an `Action` is identified, execute the corresponding tool from the `TOOLS` dictionary.
6.  Append the `Observation` to the `scratchpad`.
7.  Continue the loop until a `Final Answer` is provided or a maximum number of steps is reached.

Implement the `Agent` class below and then run it with the query: "What is the capital of France?"


In [ ]:
class Agent:
    """A simple ReAct agent that interacts with an LLM and tools."""
    def __init__(self, tools: dict, llm_predict_func):
        self.tools = tools
        self.llm_predict = llm_predict_func
        self.max_steps = 5 # Limit the number of ReAct steps to prevent infinite loops

    def _parse_llm_output(self, output: str) -> dict:
        """Parses the LLM's output to extract Thought, Action, Action Input, or Final Answer."""
        parsed_output = {}
        
        # Regex to find Thought, Action, Action Input, and Final Answer
        thought_match = re.search(r"Thought: (.*?)\n", output, re.DOTALL)
        action_match = re.search(r"Action: (.*?)\n", output, re.DOTALL)
        action_input_match = re.search(r"Action Input: (.*?)(?:\n|$)", output, re.DOTALL)
        final_answer_match = re.search(r"Final Answer: (.*?)(?:\n|$)", output, re.DOTALL)

        if thought_match: 
            parsed_output["thought"] = thought_match.group(1).strip()
        if action_match:
            parsed_output["action"] = action_match.group(1).strip()
        if action_input_match:
            parsed_output["action_input"] = action_input_match.group(1).strip()
        if final_answer_match:
            parsed_output["final_answer"] = final_answer_match.group(1).strip()
            
        return parsed_output

    def run(self, query: str) -> str:
        """Executes the ReAct loop to answer a query."""
        scratchpad = f"Question: {query}\n"
        print(f"\n--- Agent Starting for Query: '{query}' ---")

        for step in range(self.max_steps):
            print(f"\n--- Step {step + 1} ---")
            
            # 1. Call LLM with current scratchpad
            llm_response = self.llm_predict(scratchpad)
            parsed_response = self._parse_llm_output(llm_response)

            # Add Thought to scratchpad
            if "thought" in parsed_response:
                scratchpad += f"Thought: {parsed_response['thought']}\n"
                print(f"Thought: {parsed_response['thought']}")
            else:
                print("Warning: LLM response missing 'Thought'.")

            # 2. Check for Final Answer
            if "final_answer" in parsed_response:
                final_answer = parsed_response["final_answer"]
                scratchpad += f"Final Answer: {final_answer}\n"
                print(f"Final Answer: {final_answer}")
                print("--- Agent Finished ---")
                return final_answer

            # 3. Check for Action and execute tool
            if "action" in parsed_response and "action_input" in parsed_response:
                action = parsed_response["action"]
                action_input = parsed_response["action_input"]
                
                scratchpad += f"Action: {action}\n"
                scratchpad += f"Action Input: {action_input}\n"
                print(f"Action: {action}")
                print(f"Action Input: {action_input}")

                if action in self.tools:
                    tool_instance = self.tools[action]
                    observation = tool_instance.run(action_input)
                    scratchpad += f"Observation: {observation}\n"
                    print(f"Observation: {observation}")
                else:
                    observation = f"Error: Unknown tool '{action}'. Available tools: {list(self.tools.keys())}"
                    scratchpad += f"Observation: {observation}\n"
                    print(f"Observation: {observation}")
            else:
                # If no final answer and no valid action, the agent is stuck or LLM output is malformed
                print("Warning: LLM response missing 'Action' or 'Action Input'. Stopping.")
                break

        print(f"\n--- Agent Halted: Max steps ({self.max_steps}) reached or unable to proceed. ---")
        return "Agent could not find a final answer within the allowed steps."


# --- Instantiate and Run the Agent ---

# Create an instance of our agent
my_agent = Agent(tools=TOOLS, llm_predict_func=mock_llm_predict)

# Run the agent with a query
query = "What is the capital of France?"
final_result = my_agent.run(query)

print(f"\nFinal Agent Result: {final_result}")

# Test with another query (optional, to see how it handles unknown)
# print("\n" + "="*50 + "\n")
# query_unknown = "What is the capital of Mars?"
# final_result_unknown = my_agent.run(query_unknown)
# print(f"\nFinal Agent Result for '{query_unknown}': {final_result_unknown}")
